In [11]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import folium

In [35]:
# Load and project data
gdf = gpd.read_file("pilares.geojson")
gdf = gdf.to_crs(epsg=32614)
gdf.head(1)

,nombre,alcaldia,direccion,latitud,longitud,horario,geometry
0,100 Metros,Gustavo A Madero,"Av Cien Metros y Anillo Periferico S/N , Santi...",19.526523,-99.158897,De 09:00 a 19:00,POINT (483329.395 2159095.455)


In [3]:
# Create buffer for bounding box
buffers = gdf.copy()
buffers['geometry'] = buffers.buffer(1250)
bb = buffers.to_crs(4326).total_bounds.tolist()

In [4]:
# Download network (choose appropriate type)
G = ox.graph.graph_from_bbox(
    bb, 
    network_type="all",  # or "drive" or "all"
    simplify=False
)
print(f"{len(G.nodes)} nodos, {len(G.edges)} aristas")

864800 nodos, 1739394 aristas


In [5]:
# Project to UTM
G = ox.project_graph(G, to_crs=32614)
# Find nearest nodes (make sure gdf is in 32614)
nodos_cercanos = ox.distance.nearest_nodes(
    G,
    gdf.geometry.x,
    gdf.geometry.y
)
# Use only unique nodes
nodos_unicos = pd.unique(nodos_cercanos)
print(f"{len(nodos_unicos)} nodos únicos de {len(nodos_cercanos)}")

303 nodos únicos de 303


In [ ]:
# Create isodistance edges
dist_fija = 750  # meters

partes = []
for i, node in enumerate(nodos_unicos):
    try:
        subgraph = ox.truncate.truncate_graph_dist(G, node, dist_fija)
        edges = ox.convert.graph_to_gdfs(subgraph, nodes=False)
        edges["node"] = node
        partes.append(edges[["node",,'nombre' "geometry"]])
    except Exception as e:
        print(f"Error en nodo {node}: {e}")

iso_edges = gpd.GeoDataFrame(
    pd.concat(partes, ignore_index=True),
    crs=32614  # Already in correct CRS
)
print("Número total de aristas isodistancia:", len(iso_edges))
iso_edges.head()

        node                                           geometry
0  268533822  LINESTRING (483333.897 2159097.045, 483335.966...
1  268533822  LINESTRING (483333.897 2159097.045, 483320.425...
2  268533822  LINESTRING (483335.966 2159110.2, 483334.349 2...
3  268533822  LINESTRING (483320.425 2159129.125, 483312.119...
4  268533822  LINESTRING (483312.119 2159144.314, 483302.96 ...


In [41]:
# Agregar columnas de atributos desde gdf original
# Create a mapping from node to nombre
node_to_nombre = {}
for i, node in enumerate(nodos_cercanos):
    if node not in node_to_nombre:  # Only first occurrence
        node_to_nombre[node] = gdf.iloc[i]["nombre"]

# Add nombre column to iso_edges
iso_edges["nombre"] = iso_edges["node"].map(node_to_nombre)

# Check result
iso_edges.head()

,node,geometry,nombre
0,268533822,"LINESTRING (483333.897 2159097.045, 483335.966...",100 Metros
1,268533822,"LINESTRING (483333.897 2159097.045, 483320.425...",100 Metros
2,268533822,"LINESTRING (483335.966 2159110.2, 483334.349 2...",100 Metros
3,268533822,"LINESTRING (483320.425 2159129.125, 483312.119...",100 Metros
4,268533822,"LINESTRING (483312.119 2159144.314, 483302.96 ...",100 Metros


In [37]:
iso_edges.to_file("datos/calles_pilares_isodistancia_750m.geojson")

In [38]:
iso_dissolved = iso_edges.dissolve(by='node').reset_index()

iso_dissolved.to_file("datos/calles_pilares_isodistancia_750m_dissolved.geojson")
iso_dissolved.head(1)

,node,geometry,nombre
0,30594067,"MULTILINESTRING ((482807.711 2146866.108, 4827...",Benito Juárez Y El Liberalismo Mexicano


In [ ]:
iso_polygons = iso_edges.dissolve(by='node').reset_index()
iso_polygons.geometry = iso_polygons.geometry.buffer(10).convex_hull
iso_polygons.to_file("datos/isodistancia_750m_pilares.geojson")

iso_polygons.head(1)

,node,geometry,nombre
0,30594067,"POLYGON ((483148.725 2146147.127, 483147.744 2...",Benito Juárez Y El Liberalismo Mexicano


In [47]:
smooth_distance = 20  # meters (adjust as needed)

iso_polygons_smooth = iso_polygons.copy()
iso_polygons_smooth['geometry'] = (
    iso_polygons_smooth.geometry
    .buffer(-smooth_distance)  # Erode
    .buffer(smooth_distance * 2)  # Expand more
    .buffer(-smooth_distance)  # Erode back to original size
)

In [49]:
# Visualización
# Prepare data in WGS84 for Folium
gdf_wgs84 = gdf.to_crs(epsg=4326)
iso_polygons_wgs84 = iso_polygons_smooth.to_crs(epsg=4326)

# Create base map centered on Mexico City
center = [gdf_wgs84.geometry.y.mean(), gdf_wgs84.geometry.x.mean()]

m = folium.Map(
    location=center,
    zoom_start=12,
    tiles='CartoDB positron'
)

# Add isodistance polygons with styling
folium.GeoJson(
    iso_polygons_wgs84,
    name='Isodistancias (750m)',
    style_function=lambda x: {
        'fillColor': '#3388ff',
        'color': '#0066cc',
        'weight': 2,
        'fillOpacity': 0.3,
        'opacity': 0.8
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['nombre'], 
        aliases=['Pilar:'],
        localize=True
    )
).add_to(m)

# Add Pilares POIs
for idx, row in gdf_wgs84.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=8,
        popup=folium.Popup(f"<b>Pilar {idx}</b>", max_width=200),
        tooltip=f"Pilar {idx}",
        color='darkred',
        fill=True,
        fillColor='red',
        fillOpacity=0.8,
        weight=2
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Save and display
m.save('pilares_isodistancias.html')
print("Mapa guardado exitosamente!")
m

Mapa guardado exitosamente!


In [15]:
m.save('pilares_isodistancias.html')